# Frozen DistilBERT Router: Single Strategy Classifier

This notebook trains a BERT-like Router that maps a query to one strategy class such as `(2,3)`.

Configuration:
- Model: `distilbert-base-uncased`
- Encoder: frozen
- Trainable layers: dropout + linear classifier head
- Max sequence length: 64
- Batch size: 16
- Epochs: 30
- Learning rate: 2e-5
- Weight decay: 0.01
- Dropout: 0.2
- Loss: weighted cross entropy
- Evaluation: 5-fold CV, roughly 80/20 per fold, no oversampling

Note: several strategy classes have fewer than 5 examples, so strict `StratifiedKFold` is not mathematically possible. The custom splitter below distributes examples by class across folds when it can, without duplicating or leaking data.

In [19]:
# If needed, install dependencies first:
# !pip install torch transformers scikit-learn pandas tqdm

import json
import math
import random
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

In [20]:
CONFIG = {
    "data_path": "/content/clean_router_training_data.jsonl",
    "model_name": "distilbert-base-uncased",
    "max_length": 64,
    "batch_size": 16,
    "epochs": 30,
    "learning_rate": 1e-3,
    "weight_decay": 0.01,
    "dropout": 0.2,
    "n_splits": 5,
    "early_stopping_patience": 5,
    "seed": 42,
}

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
device

device(type='cuda')

## Load Query-Strategy Data

The training file should contain one JSON object per line:

```json
{"query": "...", "strategy": "(2,3)"}
```

If your file still has extra fields, this notebook ignores them.

In [21]:
def parse_strategy(strategy: str) -> tuple[int, int]:
    strategy = strategy.strip()
    if not (strategy.startswith("(") and strategy.endswith(")")):
        raise ValueError(f"Bad strategy format: {strategy}")
    left, right = strategy[1:-1].split(",")
    return int(left), int(right)

data_path = Path(CONFIG["data_path"])
rows = []
with data_path.open("r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        obj = json.loads(line)
        rows.append({"query": obj["query"], "strategy": obj["strategy"]})

df = pd.DataFrame(rows)
df[["depth", "width"]] = df["strategy"].apply(lambda s: pd.Series(parse_strategy(s)))

strategy_order = sorted(df["strategy"].unique(), key=parse_strategy)
label2id = {label: idx for idx, label in enumerate(strategy_order)}
id2label = {idx: label for label, idx in label2id.items()}
strategy_to_tuple = {label: parse_strategy(label) for label in strategy_order}

df["label"] = df["strategy"].map(label2id)
num_labels = len(label2id)

print(f"Examples: {len(df)}")
print(f"Strategy classes: {num_labels}")
display(df.head())
display(df["strategy"].value_counts().rename_axis("strategy").reset_index(name="count"))

Examples: 127
Strategy classes: 16


,query,strategy,depth,width,label
0,what continent is australia in,"(1,2)",1,2,1
1,where does missouri river end,"(1,1)",1,1,0
2,who wrote the book of st. john,"(1,1)",1,1,0
3,what illness does michael j fox have,"(1,1)",1,1,0
4,what did john howard study at university,"(4,1)",4,1,12


,strategy,count
0,"(1,1)",43
1,"(2,1)",20
2,"(1,2)",13
3,"(3,1)",10
4,"(2,3)",7
5,"(2,5)",7
6,"(1,3)",4
7,"(4,1)",4
8,"(1,5)",4
9,"(4,3)",4


## Fold Splitter

This is a no-oversampling splitter. It distributes each strategy class across folds round-robin. Classes with fewer than 5 examples necessarily appear in fewer than 5 validation folds.

In [22]:
def make_stratified_like_folds(labels, n_splits: int, seed: int):
    rng = np.random.default_rng(seed)
    label_to_indices = defaultdict(list)
    for idx, label in enumerate(labels):
        label_to_indices[int(label)].append(idx)

    folds = [[] for _ in range(n_splits)]
    for label in sorted(label_to_indices):
        indices = np.array(label_to_indices[label])
        rng.shuffle(indices)
        offset = label % n_splits
        for position, idx in enumerate(indices):
            folds[(offset + position) % n_splits].append(int(idx))

    all_indices = set(range(len(labels)))
    splits = []
    for fold_id, val_indices in enumerate(folds):
        val_indices = sorted(val_indices)
        train_indices = sorted(all_indices - set(val_indices))
        splits.append((train_indices, val_indices))
    return splits

splits = make_stratified_like_folds(df["label"].tolist(), CONFIG["n_splits"], CONFIG["seed"])
for fold_id, (train_idx, val_idx) in enumerate(splits, start=1):
    print(f"Fold {fold_id}: train={len(train_idx)}, val={len(val_idx)}")
    print("  val distribution:", dict(Counter(df.iloc[val_idx]["strategy"])))

Fold 1: train=101, val=26
  val distribution: {'(1,2)': 2, '(1,1)': 9, '(3,1)': 2, '(1,3)': 1, '(2,1)': 4, '(2,2)': 1, '(4,5)': 1, '(4,1)': 1, '(2,3)': 1, '(1,5)': 1, '(2,5)': 1, '(3,3)': 1, '(4,3)': 1}
Fold 2: train=100, val=27
  val distribution: {'(2,3)': 2, '(4,3)': 1, '(1,2)': 3, '(3,5)': 1, '(1,1)': 9, '(1,5)': 1, '(3,1)': 2, '(2,1)': 4, '(2,5)': 1, '(3,3)': 1, '(2,2)': 1, '(4,5)': 1}
Fold 3: train=100, val=27
  val distribution: {'(1,1)': 9, '(4,1)': 1, '(3,1)': 2, '(2,3)': 2, '(2,1)': 4, '(1,2)': 3, '(2,5)': 2, '(2,2)': 1, '(1,3)': 1, '(4,5)': 1, '(4,3)': 1}
Fold 4: train=103, val=24
  val distribution: {'(1,1)': 8, '(3,1)': 2, '(1,2)': 3, '(2,3)': 1, '(1,3)': 1, '(2,5)': 2, '(2,1)': 4, '(4,2)': 1, '(1,5)': 1, '(4,1)': 1}
Fold 5: train=104, val=23
  val distribution: {'(1,3)': 1, '(1,5)': 1, '(1,2)': 2, '(1,1)': 8, '(2,3)': 1, '(2,1)': 4, '(3,1)': 2, '(2,5)': 1, '(3,2)': 1, '(4,1)': 1, '(4,3)': 1}


## Dataset and Model

In [23]:
class RouterDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, tokenizer, max_length: int):
        self.queries = frame["query"].tolist()
        self.labels = frame["label"].astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.queries)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.queries[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }


class FrozenDistilBertStrategyClassifier(nn.Module):
    def __init__(self, model_name: str, num_labels: int, dropout: float):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        for param in self.encoder.parameters():
            param.requires_grad = False

        hidden_size = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0]
        logits = self.classifier(self.dropout(cls_embedding))
        return logits


def make_class_weights(train_labels, num_labels: int) -> torch.Tensor:
    counts = np.bincount(train_labels, minlength=num_labels)
    total = counts.sum()
    weights = np.zeros(num_labels, dtype=np.float32)
    present = counts > 0
    weights[present] = total / (present.sum() * counts[present])
    return torch.tensor(weights, dtype=torch.float32)

## Training and Evaluation Helpers

In [24]:
def train_one_epoch(model, loader, optimizer, loss_fn):
    model.train()
    total_loss = 0.0
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * labels.size(0)

    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(logits, labels)
        preds = logits.argmax(dim=-1)

        total_loss += loss.item() * labels.size(0)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    metrics = compute_metrics(all_labels, all_preds)
    metrics["loss"] = total_loss / len(loader.dataset)
    return metrics, all_labels, all_preds


def compute_metrics(labels, preds):
    gold_strategies = [id2label[i] for i in labels]
    pred_strategies = [id2label[i] for i in preds]

    gold_tuples = [strategy_to_tuple[s] for s in gold_strategies]
    pred_tuples = [strategy_to_tuple[s] for s in pred_strategies]

    gold_depth = [x[0] for x in gold_tuples]
    pred_depth = [x[0] for x in pred_tuples]
    gold_width = [x[1] for x in gold_tuples]
    pred_width = [x[1] for x in pred_tuples]

    depth_abs_error = np.mean([abs(a - b) for a, b in zip(gold_depth, pred_depth)])
    width_abs_error = np.mean([abs(a - b) for a, b in zip(gold_width, pred_width)])
    cost_sensitive_error = np.mean([
        abs(gd - pd) + 0.5 * abs(gw - pw)
        for (gd, gw), (pd, pw) in zip(gold_tuples, pred_tuples)
    ])

    return {
        "exact_accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "depth_accuracy": accuracy_score(gold_depth, pred_depth),
        "width_accuracy": accuracy_score(gold_width, pred_width),
        "depth_abs_error": depth_abs_error,
        "width_abs_error": width_abs_error,
        "cost_sensitive_error": cost_sensitive_error,
    }

## 5-Fold Cross-Validation

In [25]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])

fold_results = []
all_predictions = []

for fold_id, (train_idx, val_idx) in enumerate(splits, start=1):
    print(f"\n===== Fold {fold_id}/{CONFIG['n_splits']} =====")
    set_seed(CONFIG["seed"] + fold_id)

    train_df = df.iloc[train_idx].reset_index(drop=True)
    val_df = df.iloc[val_idx].reset_index(drop=True)

    missing_in_train = sorted(set(df["label"]) - set(train_df["label"]))
    if missing_in_train:
        print("Classes absent from this training fold:", [id2label[i] for i in missing_in_train])

    train_dataset = RouterDataset(train_df, tokenizer, CONFIG["max_length"])
    val_dataset = RouterDataset(val_df, tokenizer, CONFIG["max_length"])
    train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=CONFIG["batch_size"], shuffle=False)

    model = FrozenDistilBertStrategyClassifier(
        CONFIG["model_name"],
        num_labels=num_labels,
        dropout=CONFIG["dropout"],
    ).to(device)

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(
        trainable_params,
        lr=CONFIG["learning_rate"],
        weight_decay=CONFIG["weight_decay"],
    )
    #total_steps = len(train_loader) * CONFIG["epochs"]
    #scheduler = get_linear_schedule_with_warmup(
        #optimizer,
        #num_warmup_steps=0,
        #num_training_steps=total_steps,
    #)

    class_weights = make_class_weights(train_df["label"].tolist(), num_labels).to(device)
    loss_fn = nn.CrossEntropyLoss() # loss_fn = nn.CrossEntropyLoss(weight=class_weights)
    best_val_loss = math.inf
    best_state = None
    best_epoch = 0
    patience_used = 0

    for epoch in range(1, CONFIG["epochs"] + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn)
        val_metrics, _, _ = evaluate(model, val_loader, loss_fn)

        print(
            f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | "
            f"val_loss={val_metrics['loss']:.4f} | "
            f"exact_acc={val_metrics['exact_accuracy']:.3f} | "
            f"depth_acc={val_metrics['depth_accuracy']:.3f} | "
            f"width_acc={val_metrics['width_accuracy']:.3f}"
        )

        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_epoch = epoch
            patience_used = 0
        else:
            patience_used += 1
            if patience_used >= CONFIG["early_stopping_patience"]:
                print(f"Early stopping at epoch {epoch}.")
                break

    model.load_state_dict(best_state)
    final_metrics, labels, preds = evaluate(model, val_loader, loss_fn)
    final_metrics["fold"] = fold_id
    final_metrics["best_epoch"] = best_epoch
    fold_results.append(final_metrics)

    for row, gold, pred in zip(val_df.to_dict("records"), labels, preds):
        all_predictions.append({
            "fold": fold_id,
            "query": row["query"],
            "gold_strategy": id2label[gold],
            "pred_strategy": id2label[pred],
            "correct": gold == pred,
        })

results_df = pd.DataFrame(fold_results)
predictions_df = pd.DataFrame(all_predictions)
display(results_df)


===== Fold 1/5 =====


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 01 | train_loss=2.6403 | val_loss=2.3687 | exact_acc=0.346 | depth_acc=0.500 | width_acc=0.615
Epoch 02 | train_loss=2.2644 | val_loss=2.2444 | exact_acc=0.346 | depth_acc=0.500 | width_acc=0.615
Epoch 03 | train_loss=2.1446 | val_loss=2.2556 | exact_acc=0.346 | depth_acc=0.500 | width_acc=0.615
Epoch 04 | train_loss=2.1448 | val_loss=2.2577 | exact_acc=0.346 | depth_acc=0.500 | width_acc=0.615
Epoch 05 | train_loss=2.0766 | val_loss=2.2497 | exact_acc=0.346 | depth_acc=0.500 | width_acc=0.615
Epoch 06 | train_loss=2.0604 | val_loss=2.2528 | exact_acc=0.346 | depth_acc=0.500 | width_acc=0.615
Epoch 07 | train_loss=1.9906 | val_loss=2.2514 | exact_acc=0.346 | depth_acc=0.500 | width_acc=0.615
Early stopping at epoch 7.

===== Fold 2/5 =====
Classes absent from this training fold: ['(3,5)']


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

Epoch 01 | train_loss=2.5989 | val_loss=2.2926 | exact_acc=0.333 | depth_acc=0.542 | width_acc=0.625
Epoch 02 | train_loss=2.2360 | val_loss=2.2166 | exact_acc=0.333 | depth_acc=0.542 | width_acc=0.625
Epoch 03 | train_loss=2.1700 | val_loss=2.2379 | exact_acc=0.333 | depth_acc=0.542 | width_acc=0.625
Epoch 04 | train_loss=2.1550 | val_loss=2.2459 | exact_acc=0.333 | depth_acc=0.542 | width_acc=0.625
Epoch 05 | train_loss=2.1095 | val_loss=2.2600 | exact_acc=0.333 | depth_acc=0.542 | width_acc=0.625
Epoch 06 | train_loss=2.0518 | val_loss=2.2613 | exact_acc=0.333 | depth_acc=0.542 | width_acc=0.625
Epoch 07 | train_loss=2.0431 | val_loss=2.2818 | exact_acc=0.333 | depth_acc=0.542 | width_acc=0.625
Early stopping at epoch 7.

===== Fold 5/5 =====
Classes absent from this training fold: ['(3,2)']


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 01 | train_loss=2.6204 | val_loss=2.3268 | exact_acc=0.348 | depth_acc=0.522 | width_acc=0.652
Epoch 02 | train_loss=2.2296 | val_loss=2.1954 | exact_acc=0.348 | depth_acc=0.522 | width_acc=0.652
Epoch 03 | train_loss=2.1704 | val_loss=2.2257 | exact_acc=0.348 | depth_acc=0.522 | width_acc=0.652
Epoch 04 | train_loss=2.1312 | val_loss=2.2512 | exact_acc=0.348 | depth_acc=0.522 | width_acc=0.652
Epoch 05 | train_loss=2.0603 | val_loss=2.2614 | exact_acc=0.348 | depth_acc=0.522 | width_acc=0.652
Epoch 06 | train_loss=2.0889 | val_loss=2.2745 | exact_acc=0.348 | depth_acc=0.522 | width_acc=0.652
Epoch 07 | train_loss=2.0365 | val_loss=2.2695 | exact_acc=0.348 | depth_acc=0.522 | width_acc=0.652
Early stopping at epoch 7.


,exact_accuracy,macro_f1,depth_accuracy,width_accuracy,depth_abs_error,width_abs_error,cost_sensitive_error,loss,fold,best_epoch
0,0.346154,0.039560,0.500000,0.615385,0.846154,0.884615,1.288462,2.244409,1,2
1,0.333333,0.041667,0.481481,0.555556,0.814815,1.037037,1.333333,2.318930,2,2
2,0.333333,0.045455,0.481481,0.592593,0.814815,0.888889,1.259259,2.170641,3,8
3,0.333333,0.050000,0.541667,0.625000,0.708333,0.833333,1.125000,2.216642,4,2
4,0.347826,0.046921,0.521739,0.652174,0.782609,0.739130,1.152174,2.195415,5,2


## Cross-Validation Summary

In [26]:
metric_cols = [
    "exact_accuracy",
    "macro_f1",
    "depth_accuracy",
    "width_accuracy",
    "depth_abs_error",
    "width_abs_error",
    "cost_sensitive_error",
    "loss",
]

summary = pd.DataFrame({
    "mean": results_df[metric_cols].mean(),
    "std": results_df[metric_cols].std(ddof=1),
})
display(summary)

results_df.to_csv("/content/distilbert_frozen_strategy_classifier_cv_results.csv", index=False)
predictions_df.to_csv("/content/distilbert_frozen_strategy_classifier_cv_predictions.csv", index=False)

print("Saved fold metrics to distilbert_frozen_strategy_classifier_cv_results.csv")
print("Saved validation predictions to distilbert_frozen_strategy_classifier_cv_predictions.csv")

,mean,std
exact_accuracy,0.338796,0.007503
macro_f1,0.044720,0.004159
depth_accuracy,0.505274,0.026246
width_accuracy,0.608141,0.036341
depth_abs_error,0.793345,0.052566
width_abs_error,0.876601,0.108080
cost_sensitive_error,1.231646,0.089471
loss,2.229207,0.057033


Saved fold metrics to distilbert_frozen_strategy_classifier_cv_results.csv
Saved validation predictions to distilbert_frozen_strategy_classifier_cv_predictions.csv


## Inspect Mistakes

In [27]:
mistakes = predictions_df[~predictions_df["correct"]].copy()
display(mistakes.head(30))

display(
    predictions_df.groupby(["gold_strategy", "pred_strategy"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(30)
)

,fold,query,gold_strategy,pred_strategy,correct
0,1,what continent is australia in,"(1,2)","(1,1)",False
5,1,who plays lola bunny on the looney tunes show,"(3,1)","(1,1)",False
6,1,what instruments does john williams use,"(1,3)","(1,1)",False
8,1,who played daniel larusso,"(2,1)","(1,1)",False
9,1,who was king or queen after victoria,"(2,1)","(1,1)",False
11,1,what club did aguero play for before man city,"(2,2)","(1,1)",False
12,1,where is rome italy located on a map,"(4,5)","(1,1)",False
13,1,Is Deval Patrick a district representative at ...,"(3,1)","(1,1)",False
15,1,What are the sons' names of the artist who had...,"(4,1)","(1,1)",False
16,1,Which movie did Matt Bomer star in that had Sa...,"(2,3)","(1,1)",False


,gold_strategy,pred_strategy,count
0,"(1,1)","(1,1)",43
4,"(2,1)","(1,1)",20
1,"(1,2)","(1,1)",13
8,"(3,1)","(1,1)",10
7,"(2,5)","(1,1)",7
6,"(2,3)","(1,1)",7
3,"(1,5)","(1,1)",4
2,"(1,3)","(1,1)",4
14,"(4,3)","(1,1)",4
12,"(4,1)","(1,1)",4
